### Train Insect Detector + Pollinator Classifier

Trains the two models used by the **two_stage** pipeline from scratch:
1. **Binary classifier** (`binary_best.pth`) — EfficientNet-B2, insect vs background
2. **Group classifier** (`4group_insectnet.pth`) — InsectNet, 4-class: bumblebee / fly / butterfly / other

**Input** — `data/training/annotated_crops/{bumblebee,fly,butterfly,other,background}/` (each folder must contain images)  
**Output** — `outputs/training/model_runs/{RUN_NAME}_{ts}/` (logs + curves) · **overwrites** `models/binary_best.pth` and `models/4group_insectnet.pth`

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `BACKBONE` | `ValueError` if not `'insectnet'` or `'efficientnet'` |
| `BASE_DIR` (Cell 1) | Everything fails — change only when running locally |

**Optional (Cell 2):** `EPOCHS_BINARY` / `EPOCHS_S1` (default 20), `EPOCHS_S2` (default 0 — skip Stage 2, recommended for InsectNet), `LR_S1` (1e-3), `LR_S2` (1e-4), `BG_RATIO` (3), `BATCH` (32), `WEB_DIR` (None — set if web-scraped images are available)


##### Cell 1 — Environment  ← edit `BASE_DIR` for local runs
Sets all paths.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print('✓ Already extracted')
    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    BASE_DIR   = Path('/Users/lianshi/Downloads/bachelor thesis'
                    '/automated-ecological-image-analysis'
                    '/ml-pipelines/notebooks/pollinator-classification')
    DRIVE_BASE = BASE_DIR

MODEL_DIR   = BASE_DIR / 'models'
LABELED_DIR = BASE_DIR / 'data' / 'training' / 'annotated_crops'
INSECTNET_W = BASE_DIR / 'InsectNet' / 'model.pth'
WEB_IMG_DIR = BASE_DIR / 'data' / 'web_images'

# Training outputs go to local SSD on Colab (fast); saved to Drive after training.
LOCAL_TRAINING = Path('/content/outputs/training') if IN_COLAB else BASE_DIR / 'outputs' / 'training'
LOCAL_TRAINING.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET_W: {INSECTNET_W}  exists={INSECTNET_W.exists()}')
print(f'WEB_IMG_DIR: {WEB_IMG_DIR}  exists={WEB_IMG_DIR.exists()}')



##### Cell 2 — Config  ← **edit before training**
Set backbone, epochs, LR, and batch size.

In [ ]:
# ── Which sub-datasets to use ──────────────────────────────────────
# Leave [] to use ALL sub-folders automatically (excluding 'progress').
# Or list specific ones: ['labeled_ls', 'labeled_mb']
DATASETS = []

RUN_NAME     = 'binary_group'   # ← label appended to timestamp

CLASSES_4      = ['bumblebee','fly','butterfly','other']
CLASSES_BINARY = ['background','insect']
INSECT_FOLDERS = ['bumblebee','fly','butterfly','other']

# Folder name -> canonical class  (handles legacy naming)
ALIAS_4 = {
    'bumblebee':'bumblebee', 'fly':'fly',
    'butterfly':'butterfly', 'butterfly_moth':'butterfly',
    'other':'other',
}

IMG_SIZE       = 224
BATCH          = 32
EPOCHS_BINARY  = 20
EPOCHS_S1      = 20    # Stage 1: web + arctic combined
EPOCHS_S2      = 0     # Stage 2: arctic only (0 = skip, recommended for InsectNet)
LR_S1          = 1e-3
LR_S2          = 1e-4
BG_RATIO       = 3     # background:insect sampling ratio
SEED           = 42

WEB_DIR            = BASE_DIR / 'data' / 'web_images'  # iNaturalist images folder
USE_WEB_FOR_BINARY = True   # add web images as extra insect data for binary classifier
WEB_ALIAS_4 = {
    'bumblebee':'bumblebee','fly':'fly',
    'butterfly':'butterfly','butterfly_moth':'butterfly','other':'other',
}

print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET  : {INSECTNET_W}  exists={INSECTNET_W.exists()}')


##### Cell 2b — Output paths
Builds the timestamped run directory. No edits needed.

In [ ]:
from datetime import datetime

_ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = BASE_DIR / 'outputs' / 'training' / 'model_runs' / f'{RUN_NAME}_{_ts}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

import json as _json
_json.dump({
    'run_name': RUN_NAME, 'timestamp': _ts,
    'epochs_binary': EPOCHS_BINARY, 'epochs_s1': EPOCHS_S1, 'epochs_s2': EPOCHS_S2,
    'lr_s1': LR_S1, 'lr_s2': LR_S2, 'img_size': IMG_SIZE, 'batch': BATCH,
    'classes_4': CLASSES_4, 'classes_binary': CLASSES_BINARY, 'bg_ratio': BG_RATIO,
}, open(RUN_DIR / 'config.json', 'w'), indent=2)

print(f'Run directory : {RUN_DIR}')
print('Checkpoints + curves saved here; best models also copied to models/')

# ── Resolve dataset sub-folders ─────────────────────────────────
if DATASETS:
    DATASET_DIRS = [LABELED_DIR / ds for ds in DATASETS]
else:
    DATASET_DIRS = sorted([d for d in LABELED_DIR.iterdir()
                           if d.is_dir() and d.name != 'progress'])
print(f'Datasets : {[d.name for d in DATASET_DIRS]}')


##### Cell 3 — Imports + training utilities
Loads PyTorch, model definitions, and training helpers. Just run.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')


##### Cell 3b — Plotting utilities
Defines loss/accuracy curve helpers. Just run.

In [ ]:
import shutil, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np, torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN=True
except ImportError:
    HAS_SKLEARN=False

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def letterbox(img, size):
    w,h=img.size; ms=max(w,h)
    sq=Image.new('RGB',(ms,ms),(0,0,0)); sq.paste(img,((ms-w)//2,(ms-h)//2))
    return sq.resize((size,size),Image.BILINEAR)

class CropDataset(Dataset):
    def __init__(self, samples, tf): self.s=samples; self.tf=tf
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        p,l=self.s[i]; return self.tf(Image.open(p).convert('RGB')), l

def make_tf(sz, aug=False):
    base=[T.Lambda(lambda i: letterbox(i,sz)), T.ToTensor(),
          T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
    if aug:
        base=[T.Lambda(lambda i: letterbox(i,sz)),
              T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
              T.ColorJitter(0.3,0.3,0.2,0.05)]+base[1:]
    return T.Compose(base)

def make_loader(samples, idxs, sz, batch, aug=False, weighted=True):
    sub=[samples[i] for i in idxs]; ds=CropDataset(sub,make_tf(sz,aug))
    if weighted and sub:
        labs=[s[1] for s in sub]; cnt=np.bincount(labs,minlength=max(labs)+1)
        wts=[1.0/max(1,cnt[l]) for l in labs]
        return DataLoader(ds,batch_size=batch,
                          sampler=WeightedRandomSampler(wts,len(wts)),num_workers=2,pin_memory=True)
    return DataLoader(ds,batch_size=batch,shuffle=False,num_workers=2,pin_memory=True)

def sample_bg(bg_paths, n, seed=42):
    if not n: return []
    rng=np.random.default_rng(seed)
    groups={}
    for p in bg_paths:
        parts=Path(p).stem.split('_')
        k=f'{parts[0]}_{parts[3]}_{parts[4]}' if len(parts)>4 else 'default'
        groups.setdefault(k,[]).append(p)
    q=n//len(groups); rem=n%len(groups); out=[]
    for i,(k,imgs) in enumerate(sorted(groups.items())):
        rng.shuffle(imgs); take=min(q+(1 if i<rem else 0),len(imgs))
        out.extend(imgs[:take])
    print(f'Background: {len(out)} sampled from {len(bg_paths)} available')
    return out

def stratified_split(samples, val_frac=0.2, test_frac=0.1, seed=42):
    rng=np.random.default_rng(seed); by_cls=defaultdict(list)
    for i,(_,l) in enumerate(samples): by_cls[l].append(i)
    tr,va,te=[],[],[]
    for l,idxs in by_cls.items():
        rng.shuffle(idxs); n=len(idxs)
        nva=max(1,int(n*val_frac)); nte=max(1,int(n*test_frac))
        tr.extend(idxs[nva+nte:]); va.extend(idxs[nva:nva+nte]); te.extend(idxs[:nva])
    return tr,va,te

def collect_crops(labeled_dirs, classes, alias):
    ci={c:i for i,c in enumerate(classes)}; smp=[]
    for d in labeled_dirs:
        if not Path(d).exists(): continue
        for folder,cls in alias.items():
            if cls not in ci: continue
            dd=Path(d)/folder
            if not dd.exists(): continue
            for ext in ('*.jpg','*.jpeg','*.png'):
                for p in dd.glob(ext): smp.append((p,ci[cls]))
    counts={i:0 for i in range(len(classes))}
    for _,l in smp: counts[l]+=1
    print('Crops collected:')
    for i,c in enumerate(classes): print(f'  {c:20}: {counts[i]:>5}')
    return smp, counts

def build_efficientnet(n):
    m=torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    m.classifier[-1]=nn.Linear(m.classifier[-1].in_features,n)
    nn.init.xavier_uniform_(m.classifier[-1].weight); nn.init.zeros_(m.classifier[-1].bias)
    return m

def build_insectnet(weights_path, n):
    m=torchvision.models.regnet_y_32gf(); m.fc=nn.Linear(3712,2526)
    state=torch.load(weights_path,map_location='cpu',weights_only=False)
    m.load_state_dict(state.get('model',state),strict=True)
    m.fc=nn.Linear(3712,n)
    nn.init.xavier_uniform_(m.fc.weight); nn.init.zeros_(m.fc.bias)
    for name,p in m.named_parameters():
        p.requires_grad=name.startswith('fc.') or 'trunk_output.block4' in name
    print(f'InsectNet: {sum(p.numel() for p in m.parameters() if p.requires_grad):,} trainable')
    return m

def weighted_criterion(counts, n, device):
    w=torch.tensor([1.0/max(1,counts.get(i,1)) for i in range(n)],dtype=torch.float,device=device)
    return nn.CrossEntropyLoss(weight=w/w.sum())

@torch.no_grad()
def eval_epoch(model, loader, criterion, device, classes):
    model.eval(); ls=cor=tot=0; ap,al=[],[]
    tp={i:0 for i in range(len(classes))}
    fp={i:0 for i in range(len(classes))}
    fn={i:0 for i in range(len(classes))}
    for imgs,labels in loader:
        imgs,labels=imgs.to(device),labels.to(device)
        out=model(imgs); preds=out.argmax(1)
        ls+=criterion(out,labels).item()*labels.size(0)
        cor+=(preds==labels).sum().item(); tot+=labels.size(0)
        ap.extend(preds.cpu().tolist()); al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i]+=((preds==i)&(labels==i)).sum().item()
            fp[i]+=((preds==i)&(labels!=i)).sum().item()
            fn[i]+=((preds!=i)&(labels==i)).sum().item()
    pf={}
    for i in range(len(classes)):
        p=tp[i]/max(1,tp[i]+fp[i]); r=tp[i]/max(1,tp[i]+fn[i])
        pf[classes[i]]=2*p*r/max(1e-8,p+r)
    return {'loss':ls/tot,'acc':cor/tot,'macro_f1':sum(pf.values())/max(1,len(classes)),
            'per_f1':pf,'preds':ap,'labels':al}

def run_training(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt, device, classes,
                 model_dir, in_colab):
    crit=weighted_criterion(counts,len(classes),device)
    opt=torch.optim.Adam(filter(lambda p:p.requires_grad,model.parameters()),lr=lr)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=epochs)
    best_f1=0.0; hist={'tr':[],'va':[],'f1':[]}
    hdr=(f'{'Ep':>4}  {'TrLoss':>8}  {'VaLoss':>8}  {'MacroF1':>8}  {'Acc':>6}  '
         +'  '.join(f'{c[:6]:>7}' for c in classes))
    print(f'\n{"="*65}\n{name}  epochs={epochs}  lr={lr}\n{"="*65}\n{hdr}')
    for ep in range(1,epochs+1):
        model.train(); tl=tc=tt=0
        for imgs,labels in tr_ldr:
            imgs,labels=imgs.to(device),labels.to(device)
            opt.zero_grad(); out=model(imgs); loss=crit(out,labels)
            loss.backward(); opt.step()
            tl+=loss.item()*labels.size(0); tc+=(out.argmax(1)==labels).sum().item(); tt+=labels.size(0)
        vr=eval_epoch(model,va_ldr,crit,device,classes); sched.step()
        new_best=vr['macro_f1']>best_f1
        if new_best:
            best_f1=vr['macro_f1']
            torch.save({'state_dict':model.state_dict(),'classes':classes,
                        'img_size':IMG_SIZE,'val_macro_f1':best_f1,'epoch':ep},ckpt)
        pf=vr['per_f1']
        print(f'{ep:>4}  {tl/tt:>8.4f}  {vr["loss"]:>8.4f}  {vr["macro_f1"]:>8.3f}  {vr["acc"]:>6.3f}  '
              +'  '.join(f'{pf.get(c,0):>7.3f}' for c in classes)+('  *' if new_best else ''))
        hist['tr'].append(tl/tt); hist['va'].append(vr['loss']); hist['f1'].append(vr['macro_f1'])
    print(f'\nBest macro-F1: {best_f1:.3f}  ->  {ckpt}')
    # Drive backup
    if in_colab:
        try: shutil.copy(str(ckpt),str(Path(model_dir)/Path(ckpt).name)); print('Drive backup ok')
        except Exception as e: print(f'Drive backup failed: {e}')
    # Plot
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,4))
    ax1.plot(hist['tr'],label='train'); ax1.plot(hist['va'],label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'],lw=2,label='macro F1')
    ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    plt.savefig(Path(model_dir)/f'{name}_curves.png',dpi=100); plt.close()
    return model

print('Training utilities loaded.')


##### Cell 4 — Collect and split data
Loads all annotated crops, applies class aliases, and builds train/val splits.

In [ ]:
from pathlib import Path

# ── Binary data ──────────────────────────────────────────────────
insect_paths = []
bg_paths_all = []
for ds_dir in DATASET_DIRS:
    for folder in INSECT_FOLDERS:
        for ext in ('*.jpg','*.jpeg','*.png'): insect_paths.extend((ds_dir/folder).glob(ext))
    for ext in ('*.jpg','*.jpeg','*.png'): bg_paths_all.extend((ds_dir/'background').glob(ext))

# ── Add web images to binary insect class ────────────────────────
if USE_WEB_FOR_BINARY and WEB_DIR and Path(WEB_DIR).exists():
    _web_insect = []
    for _folder in INSECT_FOLDERS:
        for _ext in ('*.jpg','*.jpeg','*.png'):
            _web_insect.extend(Path(WEB_DIR).glob(f'{_folder}/{_ext}'))
    insect_paths.extend(_web_insect)
    print(f'Web images added to insect class: {len(_web_insect)}')
else:
    print('USE_WEB_FOR_BINARY=False or WEB_DIR not found — binary uses field crops only')

bg_sampled  = sample_bg(bg_paths_all, len(insect_paths)*BG_RATIO, SEED)
binary_data = [(p,1) for p in insect_paths] + [(p,0) for p in bg_sampled]
bin_counts  = {0:len(bg_sampled), 1:len(insect_paths)}
bin_tr, bin_va, bin_te = stratified_split(binary_data, seed=SEED)
print(f'Binary: insect={bin_counts[1]}  bg={bin_counts[0]}')
print(f'Split : train={len(bin_tr)}  val={len(bin_va)}  test={len(bin_te)}')

# ── Group data ───────────────────────────────────────────────────
arctic_smp, counts_arc = collect_crops(DATASET_DIRS, CLASSES_4, ALIAS_4)
web_smp, counts_web = (collect_crops([WEB_DIR], CLASSES_4, WEB_ALIAS_4)
                       if WEB_DIR and Path(WEB_DIR).exists() else ([],{i:0 for i in range(4)}))
arc_tr, arc_va, arc_te = stratified_split(arctic_smp, seed=SEED)
web_tr, web_va, web_te = stratified_split(web_smp,    seed=SEED) if web_smp else ([],[],[])
print(f'\nGroup: arctic={len(arctic_smp)}  web={len(web_smp)}')


##### Cell 5 — Train binary classifier
Trains the insect vs background EfficientNet. Saves best weights to `models/binary_best.pth`.

In [ ]:
model_bin = build_efficientnet(2).to(DEVICE)
tr_ldr = make_loader(binary_data, bin_tr, IMG_SIZE, BATCH, aug=True)
va_ldr = make_loader(binary_data, bin_va, IMG_SIZE, BATCH)
ckpt   = RUN_DIR/'binary_best.pth'
model_bin = run_training(model_bin,'insect_detector',tr_ldr,va_ldr,
                         EPOCHS_BINARY,LR_S1,bin_counts,ckpt,DEVICE,
                         CLASSES_BINARY,RUN_DIR,IN_COLAB)
# Test eval
te_ldr = make_loader(binary_data,bin_te,IMG_SIZE,BATCH)
te = eval_epoch(model_bin,te_ldr,
                __import__('torch').nn.CrossEntropyLoss(),DEVICE,CLASSES_BINARY)
print(f'Test: MacroF1={te["macro_f1"]:.3f}  Acc={te["acc"]:.3f}')
if HAS_SKLEARN:
    print(classification_report(te['labels'],te['preds'],target_names=CLASSES_BINARY,digits=3))

# Copy best checkpoint to models/ so inference notebooks stay unchanged
import shutil
shutil.copy(ckpt, MODEL_DIR / 'binary_best.pth')
print(f'  → models/binary_best.pth updated')

# Save classification report
if HAS_SKLEARN:
    rpt = classification_report(te['labels'], te['preds'],
                                target_names=CLASSES_BINARY, digits=3)
    (RUN_DIR / 'binary_test_report.txt').write_text(
        f'binary_best  test_macro_f1={te["macro_f1"]:.3f}\n\n' + rpt)
    print('  → binary_test_report.txt saved')


##### Cell 6 — Train group classifier
Trains the 4-class InsectNet (bumblebee / fly / butterfly / other). Saves to `models/4group_insectnet.pth`.

In [ ]:
import shutil

model_g = build_insectnet(INSECTNET_W, 4).to(DEVICE)

# Stage 1: web + arctic combined
if web_smp:
    combined = arctic_smp + web_smp
    comb_tr  = arc_tr + [len(arctic_smp)+i for i in web_tr]
    comb_cnt = {i: counts_arc.get(i,0)+counts_web.get(i,0) for i in range(4)}
    tr_s1 = make_loader(combined,  comb_tr, IMG_SIZE, BATCH, aug=True)
    va_s1 = make_loader(arctic_smp+web_smp,
                        arc_va+[len(arctic_smp)+i for i in web_va], IMG_SIZE, BATCH)
else:
    comb_cnt = counts_arc
    tr_s1 = make_loader(arctic_smp, arc_tr, IMG_SIZE, BATCH, aug=True)
    va_s1 = make_loader(arctic_smp, arc_va, IMG_SIZE, BATCH)

ckpt_s1 = RUN_DIR/'4group_insectnet_stage1.pth'
model_g = run_training(model_g,'pollinator_classifier_s1',tr_s1,va_s1,
                       EPOCHS_S1,LR_S1,comb_cnt,ckpt_s1,DEVICE,CLASSES_4,RUN_DIR,IN_COLAB)

# Stage 2: arctic fine-tune (skip by default)
ckpt_final = RUN_DIR/'4group_insectnet.pth'
if EPOCHS_S2==0:
    print('Stage2=0 — copying Stage 1 as final model.')
    shutil.copy(ckpt_s1, ckpt_final)
else:
    model_g.load_state_dict(
        __import__('torch').load(ckpt_s1,map_location=DEVICE,weights_only=False)['state_dict'])
    tr_s2 = make_loader(arctic_smp, arc_tr, IMG_SIZE, BATCH, aug=True)
    va_s2 = make_loader(arctic_smp, arc_va, IMG_SIZE, BATCH)
    model_g = run_training(model_g,'pollinator_classifier_s2',tr_s2,va_s2,
                           EPOCHS_S2,LR_S2,counts_arc,ckpt_final,DEVICE,CLASSES_4,RUN_DIR,IN_COLAB)

# Final test eval
model_g.load_state_dict(
    __import__('torch').load(ckpt_final,map_location=DEVICE,weights_only=False)['state_dict'])
te_arc = make_loader(arctic_smp, arc_te, IMG_SIZE, BATCH)
ta = eval_epoch(model_g,te_arc,
                __import__('torch').nn.CrossEntropyLoss(),DEVICE,CLASSES_4)
print(f'\nTest Arctic: MacroF1={ta["macro_f1"]:.3f}  Acc={ta["acc"]:.3f}')
if HAS_SKLEARN:
    print(classification_report(ta['labels'],ta['preds'],target_names=CLASSES_4,digits=3))

# Copy best checkpoint to models/ so inference notebooks stay unchanged
shutil.copy(ckpt_final, MODEL_DIR / '4group_insectnet.pth')
print(f'  → models/4group_insectnet.pth updated')

# Save classification report
if HAS_SKLEARN:
    rpt = classification_report(ta['labels'], ta['preds'],
                                target_names=CLASSES_4, digits=3)
    (RUN_DIR / 'group4_test_report.txt').write_text(
        f'4group_insectnet  test_macro_f1={ta["macro_f1"]:.3f}\n\n' + rpt)
    print('  → group4_test_report.txt saved')

print(f'\nAll outputs in: {RUN_DIR}')
